### Connection + Cursor Handling

In [1]:
import pandas as pd
import psycopg2
from psycopg2.extras import execute_values
import os

try:
    conn = psycopg2.connect(
        host = "localhost",
        database = "flowcart",
        user = "postgres",
        password = os.getenv("DB_PASSWORD"),
        port = "5432"
    )
    cur = conn.cursor()
    print("Database connection successful")

except Exception as e:
        print("Error connecting to database:", e)    

Database connection successful


### Create Tables with Error Handling

In [2]:
try:
    cur.execute("CREATE SCHEMA IF NOT EXISTS flowcart;")
    cur.execute("""
    CREATE TABLE IF NOT EXISTS flowcart.products (
        product_id INT PRIMARY KEY,
        category TEXT,
        sub_category TEXT
    );
    """)

    cur.execute("""
    CREATE TABLE IF NOT EXISTS flowcart.customers (
        customer_id INT PRIMARY KEY,
        customer_name TEXT
    );
    """)

    cur.execute("""
    CREATE TABLE IF NOT EXISTS flowcart.payments (
        payment_id INT PRIMARY KEY,
        payment_mode TEXT
    );
    """)

    cur.execute("""
    CREATE TABLE IF NOT EXISTS flowcart.locations (
        location_id INT PRIMARY KEY,
        state TEXT,
        city TEXT
    );
    """)

    cur.execute("""
    CREATE TABLE IF NOT EXISTS flowcart.dates(
        date_id INT PRIMARY KEY,
        order_date DATE NOT NULL,
        year INT,
        day_name TEXT,
        month_name TEXT,  
        quarter INT                                 
        );
    """)

    cur.execute("""
    CREATE TABLE IF NOT EXISTS flowcart.orders (
        order_items_id INT PRIMARY KEY,
        order_id TEXT,
        customer_id INT REFERENCES flowcart.customers(customer_id),
        product_id INT REFERENCES flowcart.products(product_id),
        location_id INT REFERENCES flowcart.locations(location_id),
        payment_id INT REFERENCES flowcart.payments(payment_id),
        date_id INT REFERENCES flowcart.dates(date_id),
        amount FLOAT,
        quantity INT,
        profit FLOAT
    );
    """)

    conn.commit()
    print("Tables created successfully")

except Exception as e:
    conn.rollback()
    print("Error creating tables:", e)        

Tables created successfully


### Create transactional tables

In [3]:
try:
    cur.execute("CREATE SCHEMA IF NOT EXISTS transact;")

    cur.execute("""
    CREATE TABLE IF NOT EXISTS transact.payments(
        payment_id INT PRIMARY KEY NOT NULL,
        payment_mode TEXT           
    );
    """)

    cur.execute("""
    CREATE TABLE IF NOT EXISTS transact.customers(
        customer_id INT PRIMARY KEY NOT NULL,
        customer_name TEXT
    );
    """)

    cur.execute("""
    CREATE TABLE IF NOT EXISTS transact.products(
        product_id INT PRIMARY KEY NOT NULL,
        category TEXT,
        sub_category TEXT,  
        amount FLOAT    
    );
    """)

    cur.execute("""
    CREATE TABLE IF NOT EXISTS transact.locations(
        location_id INT PRIMARY KEY NOT NULL,
        city TEXT,
        state TEXT        
    );
    """)

    cur.execute("""
    CREATE TABLE IF NOT EXISTS transact.orders(
        order_items_id INT PRIMARY KEY NOT NULL,
        order_id TEXT,
        payment_id INT,  
        customer_id INT,
        product_id INT,
        location_id INT,
        order_date DATE,
        quantity INT,
        profit FLOAT        
    );
    """)
    
    conn.commit()
    print("Transactions tables created successfully")
except Exception as e:
    print("Error creating transactions table",e)





Transactions tables created successfully


### Self Truncate (prevents Duplicate Inserts)

In [4]:
try:
    cur.execute("""
    TRUNCATE TABLE flowcart.orders,
    flowcart.customers,
    flowcart.products,
    flowcart.locations,
    flowcart.payments,
    flowcart.dates,
    transact.customers,
    transact.payments,
    transact.products,
    transact.locations,
    transact.orders CASCADE;
    """)
    
    conn.commit()
    print("Tables truncated")

except Exception as e:
    conn.rollback()
    print("Error truncating tables:", e)

Tables truncated


### Insert function

In [5]:
def insert_df(table,df):
    cur = conn.cursor()
    tuples = [tuple(x) for x in df.to_numpy()]
    cols = ','.join(df.columns)
    query = f"INSERT INTO {table} ({cols}) VALUES %s"

    try:
        execute_values(cur,query,tuples)
        conn.commit()
        print(f"Successful! Inserted into {table}")
    except Exception as e:
        conn.rollback()
        print(f"Error inserting into {table}")
        print(e)
    cur.close()

### Load CSV Files

In [6]:
# loading warehouse table
customers_dim_df = pd.read_csv(r"Dataset\clean_dataset\customers.csv")
locations_dim_df = pd.read_csv(r"Dataset\clean_dataset\locations.csv")
products_dim_df = pd.read_csv(r"Dataset\clean_dataset\products.csv")
payments_dim_df = pd.read_csv(r"Dataset\clean_dataset\payments.csv")
dates_dim_df = pd.read_csv(r"Dataset\clean_dataset\dates.csv")
orders_fact_df = pd.read_csv(r"Dataset\clean_dataset\orders.csv")


# loading transactions table
customers_df = pd.read_csv(r"Dataset\transactions_data\customers.csv")
locations_df = pd.read_csv(r"Dataset\transactions_data\locations.csv")
products_df = pd.read_csv(r"Dataset\transactions_data\products.csv")
payments_df = pd.read_csv(r"Dataset\transactions_data\payments.csv")
orders_df = pd.read_csv(r"Dataset\transactions_data\orders.csv")


### Inserting in Correct Order

In [7]:
# inserting into data_warehouse
insert_df("flowcart.customers",customers_dim_df)
insert_df("flowcart.products",products_dim_df)
insert_df("flowcart.locations",locations_dim_df)
insert_df("flowcart.payments",payments_dim_df)
insert_df("flowcart.dates",dates_dim_df)
insert_df("flowcart.orders",orders_fact_df)

# inserting into the database
insert_df("transact.customers",customers_df)
insert_df("transact.payments",payments_df) 
insert_df("transact.products",products_df)
insert_df("transact.locations",locations_df)
insert_df("transact.orders",orders_df)
    
   

Successful! Inserted into flowcart.customers
Successful! Inserted into flowcart.products
Successful! Inserted into flowcart.locations
Successful! Inserted into flowcart.payments
Successful! Inserted into flowcart.dates
Successful! Inserted into flowcart.orders
Successful! Inserted into transact.customers
Successful! Inserted into transact.payments
Successful! Inserted into transact.products
Successful! Inserted into transact.locations
Successful! Inserted into transact.orders


### Close Connection

In [8]:
try:
    cur.close()
    conn.close()
    print("Connection closed")
except Exception as e:
    print("Error closing connection:",e)

Connection closed
